# Quantize Your Fine-Tuned Model

<a target="_blank" href="https://colab.research.google.com/github/Dannys0n/CS-394/blob/main/src/07/notebooks/quantize.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://github.com/Dannys0n/CS-394/raw/refs/heads/main/src/07/notebooks/quantize.ipynb">
  <img src="https://img.shields.io/badge/Download_.ipynb-blue" alt="Download .ipynb"/>
</a>

## Install dependencies

In [ ]:
!uv pip install huggingface_hub

## Clone and build llama.cpp (for running on Colab)

In [ ]:
TEMP_FOLDER = "/content/tmp"

# Create a temporary location and clone llama.cpp
!mkdir -p {TEMP_FOLDER}
!cd {TEMP_FOLDER} && git clone https://github.com/ggml-org/llama.cpp

!cd {TEMP_FOLDER}/llama.cpp && cmake -B build
!cd {TEMP_FOLDER}/llama.cpp && cmake --build build --config Release --target llama-quantize -j 8


## Run convert hf to gguf script

In [ ]:
from huggingface_hub import snapshot_download

HF_USERNAME = "Dannys0n" # Change this to your Hugging Face username
MODEL_NAME = "Qwen3-1.7B-cs2-commentators" # Change this to the repo of your merged model
MODEL_REPO = f"{HF_USERNAME}/{MODEL_NAME}"

# Working folders
MODEL_FOLDER = "/content/model"
GGUF_FOLDER = "/content/ggufs"

!mkdir -p {MODEL_FOLDER}
!mkdir -p {GGUF_FOLDER}

# Download the model from Hugging Face at full precision
snapshot_download(repo_id=MODEL_REPO, local_dir=f"{MODEL_FOLDER}", repo_type="model")

# Convert to GGUF (fp16 first)
!cd {TEMP_FOLDER}/llama.cpp && python convert_hf_to_gguf.py {MODEL_FOLDER} \
    --outfile {GGUF_FOLDER}/{MODEL_NAME}-F16.gguf \
    --outtype f16

# # Convert gp16 GGUF to Q4_K_M
!{TEMP_FOLDER}/llama.cpp/build/bin/llama-quantize {GGUF_FOLDER}/{MODEL_NAME}-F16.gguf \
    {GGUF_FOLDER}/{MODEL_NAME}-Q4_K_M.gguf \
    Q4_K_M

## Upload GGUFs to Hugging Face repo

In [ ]:
from huggingface_hub import HfApi

# Initialize the API
api = HfApi()

# Upload GGUF files to root
print("\nUploading GGUF models to root...")
api.upload_folder(
    folder_path=f"{GGUF_FOLDER}",
    repo_id=MODEL_REPO,
    repo_type="model",
    path_in_repo="",  # GGUF files are typically found on the root level of a HF repo
    commit_message="Upload GGUF quantized models"
)

print(f"\nGGUFs uploaded successfully to https://huggingface.co/{MODEL_REPO}")